# Finite-estimator uncertainty and intervention sensitivity

This offline tutorial uses the binary setosa/versicolor subset of scikit-learn's bundled Iris data and a seeded logistic-regression pipeline. It demonstrates two narrow diagnostics for output column 1, $p_1(x)=P(Y=1\mid x)$:

1. repeated finite Monte Carlo estimates of the probability drop after replacing one feature from an explicitly named empirical background; and
2. deterministic sensitivity of that probability drop to three prespecified replacement values.

The reported Student-t interval covers only the mean over the supplied seeded RNG streams. Neither the interval nor the sensitivity range is a proof of global faithfulness, robustness, causality, model quality, or deployment suitability.

## Setup

The dataset ships with scikit-learn, and every computation is bounded and CPU-only. `SEED` controls model fitting; the Monte Carlo streams are declared separately so every finite replicate is identifiable.

In [1]:
import numpy as np
import sklearn

from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from explainiverse.evaluation import (
    evaluate_intervention_sensitivity,
    run_seeded_replicates,
)

SEED = 31
np.random.seed(SEED)
print(f"scikit-learn {sklearn.__version__}; model seed {SEED}")

scikit-learn 1.7.2; model seed 31


## Fit one explicit probability-output model

We keep only Iris target labels 0 and 1. The pipeline standardizes all four features, then fits binary logistic regression. For standardized row $z$, the asserted output formula is

$$p_1(x)=\sigma(\beta_0+\beta^T z),\qquad \sigma(a)=\frac{1}{1+e^{-a}}.$$

All rows are used to fit this compact teaching model. Its predictions are not held-out performance evidence.

In [2]:
iris = load_iris()
binary_rows = np.asarray(iris.target) < 2
X = np.asarray(iris.data[binary_rows], dtype=np.float64)
y = np.asarray(iris.target[binary_rows], dtype=np.int64)
feature_names = [str(name) for name in iris.feature_names]

model = make_pipeline(
    StandardScaler(),
    LogisticRegression(solver="liblinear", random_state=SEED),
)
model.fit(X, y)

scaled = model.named_steps["standardscaler"].transform(X)
classifier = model.named_steps["logisticregression"]
logits = scaled @ classifier.coef_[0] + classifier.intercept_[0]
formula_probabilities = 1.0 / (1.0 + np.exp(-logits))
model_probabilities = model.predict_proba(X)[:, 1]
np.testing.assert_allclose(model_probabilities, formula_probabilities, rtol=0.0, atol=1e-12)
assert np.all((model_probabilities >= 0.0) & (model_probabilities <= 1.0))

instance = X[60].copy()
target_output_column = 1
original_probability = float(model.predict_proba(instance.reshape(1, -1))[0, target_output_column])
assert 0.0 <= original_probability <= 1.0
print(f"rows={len(X)}; target output column={target_output_column}; p1(x*)={original_probability:.6f}")

rows=100; target output column=1; p1(x*)=0.969604


## Repeated finite estimates

Feature $j$ is petal length. For seed $s$, each index $I_{s,k}$ is sampled uniformly with replacement from the declared 100-row binary-Iris background. With $m=64$, one replicate estimates

$$\widehat{\mu}_s=\frac{1}{m}\sum_{k=1}^{m}\left[p_1(x^*)-p_1\left(x^*_{j\leftarrow B_{I_{s,k},j}}\right)\right].$$

The score space is a class-1 probability difference and therefore lies in $[-1,1]$. Different seeds sample different finite empirical-background draws; they do not vary the model, input population, feature dependence assumption, or intervention.

In [3]:
FEATURE_INDEX = feature_names.index("petal length (cm)")
SAMPLE_COUNT = 64
REPLICATE_SEEDS = (101, 202, 303, 404, 505)

def probability_drop_estimate(*, seed):
    rng = np.random.default_rng(seed)
    sampled_rows = rng.integers(0, len(X), size=SAMPLE_COUNT)
    perturbed = np.repeat(instance.reshape(1, -1), SAMPLE_COUNT, axis=0)
    perturbed[:, FEATURE_INDEX] = X[sampled_rows, FEATURE_INDEX]
    drops = original_probability - model.predict_proba(perturbed)[:, target_output_column]
    assert drops.shape == (SAMPLE_COUNT,)
    assert np.all(np.isfinite(drops))
    assert np.all((-1.0 <= drops) & (drops <= 1.0))
    return float(np.mean(drops))

manual_replicates = [probability_drop_estimate(seed=seed) for seed in REPLICATE_SEEDS]
replicate_report = run_seeded_replicates(
    probability_drop_estimate,
    seeds=REPLICATE_SEEDS,
    sample_count=SAMPLE_COUNT,
    confidence_level=0.95,
    convergence_tolerance=0.05,
)
np.testing.assert_allclose(
    replicate_report["replicate_estimates"], manual_replicates, rtol=0.0, atol=0.0
)
assert replicate_report["seeds"] == list(REPLICATE_SEEDS)
assert replicate_report["sample_count_per_replicate"] == SAMPLE_COUNT
assert replicate_report["confidence_interval_defined"] is True
assert replicate_report["confidence_interval_kind"] == "student_t_mean_of_independent_seeded_replicates"
assert replicate_report["confidence_interval_scope"] == "mean_over_the_supplied_rng_streams_only"
assert replicate_report["convergence_diagnostic_only"] is True
assert replicate_report["finite_estimate_is_global_proof"] is False
assert replicate_report["confidence_interval"][0] <= replicate_report["estimate"] <= replicate_report["confidence_interval"][1]

fresh_replicate_report = run_seeded_replicates(
    probability_drop_estimate,
    seeds=REPLICATE_SEEDS,
    sample_count=SAMPLE_COUNT,
    confidence_level=0.95,
    convergence_tolerance=0.05,
)
assert fresh_replicate_report == replicate_report
print("replicate estimates:", [round(value, 6) for value in replicate_report["replicate_estimates"]])
print("mean and 95% t interval:", round(replicate_report["estimate"], 6), [round(value, 6) for value in replicate_report["confidence_interval"]])
print("fresh seeded report is exactly equal:", fresh_replicate_report == replicate_report)

replicate estimates: [0.0897, 0.096228, 0.090705, 0.118104, 0.072447]
mean and 95% t interval: 0.093437 [0.07305, 0.113823]
fresh seeded report is exactly equal: True


## Prespecified intervention sensitivity

Before evaluating the score, we declare three empirical-background references for petal length: its 10th percentile, median, and 90th percentile. For replacement $b$, the deterministic estimand is

$$d_b=p_1(x^*)-p_1\left(x^*_{j\leftarrow b}\right).$$

These references share one exact replacement contract, target output, feature, and background. They are plausible teaching choices, not universal defaults; replacement may create rows that are atypical under the joint feature distribution.

In [4]:
interventions = {
    "binary_iris_empirical_q10": float(np.quantile(X[:, FEATURE_INDEX], 0.10)),
    "binary_iris_empirical_median": float(np.quantile(X[:, FEATURE_INDEX], 0.50)),
    "binary_iris_empirical_q90": float(np.quantile(X[:, FEATURE_INDEX], 0.90)),
}
INTERVENTION_CONTRACT = (
    "v1; replace petal length with the named binary-Iris empirical quantile; "
    "score=original-minus-intervened class-1 probability"
)

def deterministic_probability_drop(replacement):
    perturbed = instance.copy()
    perturbed[FEATURE_INDEX] = replacement
    intervened_probability = float(
        model.predict_proba(perturbed.reshape(1, -1))[0, target_output_column]
    )
    value = original_probability - intervened_probability
    assert -1.0 <= value <= 1.0
    return value

sensitivity_report = evaluate_intervention_sensitivity(
    interventions,
    deterministic_probability_drop,
    intervention_contract=INTERVENTION_CONTRACT,
)
manual_scores = {
    name: deterministic_probability_drop(value) for name, value in interventions.items()
}
np.testing.assert_allclose(
    list(sensitivity_report["scores"].values()),
    list(manual_scores.values()),
    rtol=0.0,
    atol=0.0,
)
expected_range = max(manual_scores.values()) - min(manual_scores.values())
np.testing.assert_allclose(sensitivity_report["sensitivity_range"], expected_range, rtol=0.0, atol=1e-15)
assert sensitivity_report["intervention_contract"] == INTERVENTION_CONTRACT
assert sensitivity_report["intervention_names"] == list(interventions)
assert sensitivity_report["universal_default_claimed"] is False
assert sensitivity_report["shared_intervention_contract_required_for_comparison"] is True
assert sensitivity_report["conclusion_invariant_across_prespecified_interventions"] is False
assert min(manual_scores.values()) < 0.0 < max(manual_scores.values())

fresh_sensitivity_report = evaluate_intervention_sensitivity(
    interventions,
    deterministic_probability_drop,
    intervention_contract=INTERVENTION_CONTRACT,
)
assert fresh_sensitivity_report == sensitivity_report
print("prespecified probability drops:", {name: round(value, 6) for name, value in sensitivity_report["scores"].items()})
print("sensitivity range:", round(sensitivity_report["sensitivity_range"], 6))
print("same sign across references:", not (min(manual_scores.values()) < 0.0 < max(manual_scores.values())))
print("fresh sensitivity report is exactly equal:", fresh_sensitivity_report == sensitivity_report)

prespecified probability drops: {'binary_iris_empirical_q10': 0.212522, 'binary_iris_empirical_median': 0.056469, 'binary_iris_empirical_q90': -0.021657}
sensitivity range: 0.234179
same sign across references: False
fresh sensitivity report is exactly equal: True


## Interpretation boundaries

The assertions establish the declared formula, class-1 probability output, score range, seeded fresh-report equality, finite-replicate metadata, and exact multi-reference contract for this checkout. The sign change across the three references is evidence that this local conclusion depends on the chosen intervention—not evidence that one reference is correct.

The interval does not cover new inputs, alternative models, other backgrounds, or deployment drift. The baseline replacements do not preserve the joint feature distribution. No result here certifies explanation faithfulness, robustness, causality, fairness, model validity, or decision usefulness. Those permanent boundaries require task-specific design and evidence outside this tutorial.